# P15 — Recommandation de métiers à partir de compétences (ROME)

**Contexte métier.** France Travail, POLE de conseil en évolution professionnelle, plateformes d'orientation : prédire les métiers compatibles avec un profil de compétences, ou suggérer les compétences à acquérir pour viser un métier. Enjeu : 1 911 fiches métiers ROME reliées à 48 420 compétences.

**Objectif de l'EDA.** Construire la matrice métiers × compétences, mesurer sa structure (compétences transverses vs spécialisées), identifier la cible (grand domaine du métier) et les régresseurs (compétences binarisées).

**Données.** Référentiel ROME v4.61 — France Travail, open data (api.francetravail.fr). Licence ouverte. Le zip `?referentiel=rome` contient ~30 CSV, dont le référentiel des métiers, les items et les liens métier-compétence.

## 1. Téléchargement

Le référentiel ROME est distribué en zip via l'API open data de France Travail :

In [ ]:
import urllib.request, zipfile, os
os.makedirs("data/rome", exist_ok=True)

URL = "https://api.francetravail.fr/api-nomenclatureemploi/v1/open-data/csv?referentiel=rome"
if not os.path.exists("data/rome/unix_item_v461_utf8.csv"):
    urllib.request.urlretrieve(URL, "data/rome.zip")
    with zipfile.ZipFile("data/rome.zip") as z:
        for n in z.namelist():
            if n.startswith("unix_"):
                z.extract(n, "data/rome")
print(len(os.listdir("data/rome")), "fichiers extraits")

## 2. Chargement

Trois fichiers suffisent : les métiers (1 911), les items (35 786) et les liens métier-item (140 246). Le premier caractère du code ROME donne le grand domaine.

In [ ]:
import pandas as pd, numpy as np

R = "data/rome/unix_"
rome = pd.read_csv(R + "referentiel_code_rome_v461_utf8.csv", encoding="utf-8-sig")
liens = pd.read_csv(R + "liens_rome_referentiels_v461_utf8.csv", encoding="utf-8-sig")
item = pd.read_csv(R + "item_v461_utf8.csv", encoding="utf-8-sig")
comp = pd.read_csv(R + "referentiel_competence_v461_utf8.csv", encoding="utf-8-sig")
for x in (rome, liens, item, comp):
    x.columns = [c.strip() for c in x.columns]
li = liens.merge(item.rename(columns={"code_rubrique": "rubrique_item"}), on="code_ogr", how="left")
li = li.merge(comp[["code_ogr", "libelle_competence"]], on="code_ogr", how="left")
li["libelle"] = li["libelle_competence"].fillna(li["libelle"])
print(len(rome), "metiers,", len(item), "items,", len(li), "liens")

## 3. Analyse

Structure du graphe bipartite : nombre de compétences par métier, nombre de métiers par compétence, compétences les plus transverses, répartition par grand domaine.

In [ ]:
per_metier = li.groupby("code_rome").size()
per_comp = li.groupby("code_ogr").size()
print("Compétences par métier :", per_metier.describe(percentiles=[.1, .5, .9]).round(1).to_dict())
print("Métiers par compétence :", per_comp.describe(percentiles=[.5, .9, .99]).round(1).to_dict())
print()
top = li.groupby("libelle")["code_rome"].nunique().sort_values(ascending=False).head(10)
print("Compétences transverses :", top.to_dict())
print()
print("Métiers par grand domaine :", rome["code_rome"].str[0].value_counts().sort_index().to_dict())
print("Compétences uniques (1 seul métier) :", round((per_comp == 1).mean() * 100, 1), "%")

## 4. Résultats observés (exécution du 16/10/2026)

- n metiers : 1911
- n items : 35786
- n liens metier competence : 140246
- competences par metier stats : count : 1 911.00, mean : 73.40, std : 18.90, min : 31.00, 10% : 52.00, 50% : 71.00, 90% : 98.00, max : 210.00
- metiers par competence stats : count : 48 420.00, mean : 2.90, std : 17.00, min : 1.00, 50% : 1.00, 90% : 4.00, 99% : 26.00, max : 1 552.00
- top competences libelles : Salarié secteur privé (CDI, CDD) : 1552, Faire preuve de rigueur et de précision : 1231, Travail en journée : 1072, O…
- n metiers par grand domaine : A : 107, B : 54, C : 64, D : 140, E : 79, F : 114, G : 119, H : 280, I : 132, J : 86
- competences uniques pct : 69.30
- metiers sans competence : 0
- transitions eco dist : Emploi Blanc : 1388, Emploi stratégique pour la Transition écologique : 366, Emploi Vert : 109, Emploi Brun : 48
- emploi reglemente dist : N : 1235, nan : 399, O : 277

## 5. Variable cible et régresseurs

**Cible (classification multi-classe) : grand domaine du métier** (lettre initiale du code ROME, 16 domaines), voire la fiche ROME elle-même (1 911 classes) — prédit depuis l'ensemble des compétences d'un métier. Variante métier : recommandation par similarité cosinus dans l'espace des compétences.

**Régresseurs :**
- La matrice binaire métier × compétence (48 420 compétences, 140 246 liens) — en multi-label ou embedding
- Type d'item : savoir-faire (`code_rubrique` 1) vs savoir théorique (2)
- `emploi_reglemente`, `transition_eco` (champs du référentiel métiers)

**Pièges :**
- 48 420 colonnes pour 1 911 lignes : classification en très haute dimension — réduire (embedding, sélection par fréquence documentaire), ou viser le grand domaine
- Les compétences transverses ("Analyser", 1 552 métiers) n'apportent aucun signal de discrimination — les pénaliser comme des stopwords
- 27 % des compétences n'apparaissent que dans un seul métier : risque de mémorisation — split train/test soigné
- ROME évolue (v4.61 ici) : verrouiller la version dans le projet